In [1]:
# Kill all processes on the GPU
!fuser -v /dev/nvidia* -k

                     USER        PID ACCESS COMMAND
/dev/nvidia0:        root       2626 F...m python3
                     root       3277 F.... wandb-xpu
/dev/nvidiactl:      root       2626 F...m python3
                     root       3277 F.... wandb-xpu
/dev/nvidia-uvm:     root       2626 F...m python3
                     root       3277 F.... wandb-xpu


# Libraries

In [2]:
%%capture
!uv pip uninstall torchao torchaudio torchvision -y
!uv pip install \
    "transformers==4.53.3" \
    "peft==0.17.1" \
    "trl" \
    "accelerate" \
    "bitsandbytes" \
    "wandb"

In [3]:
import json
import math
import torch
from datetime import datetime
from pathlib import Path
from collections import OrderedDict

from transformers import AutoModelForQuestionAnswering, AutoTokenizer
from huggingface_hub import snapshot_download
from safetensors.torch import load_file

# Configurations

In [4]:
# Run configuration
SRC_LANG = 'en' # 'en' | '<any_lang>'
TGT_LANG = 'tr' # 'en' | '<any_lang>'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Model configuration
MODEL_ID = 'FacebookAI/xlm-roberta-base'
# ---
# LEGAMEX_TASK_ID = 'alxxtexxr/XLM-R-Base-squad-en-15K-s42-LegameX-LoRA-v261009193519'
# LEGAMEX_TASK_CKPT_STEP = 7800
# LEGAMEX_LANG_ID = 'alxxtexxr/XLM-R-Base-wikipedia-tr-3.5K-s42-LegameX-LoRA-v261010161203'
# LEGAMEX_LANG_CKPT_STEP = 400
# ---
# LEGAMEX_TASK_ID = 'alxxtexxr/XLM-R-Base-squad-en-15K-s42-LegameX-LoRA-v261010102224'
# LEGAMEX_TASK_CKPT_STEP = 6600
# LEGAMEX_LANG_ID = 'alxxtexxr/XLM-R-Base-wikipedia-tr-3.5K-s42-LegameX-LoRA-v261010184556'
# LEGAMEX_LANG_CKPT_STEP = 400
# ---
# LEGAMEX_TASK_ID = 'alxxtexxr/XLM-R-Base-squad-en-15K-s42-LegameX-LoRA-v261011131449'
# LEGAMEX_TASK_CKPT_STEP = 7800
# LEGAMEX_LANG_ID = 'alxxtexxr/XLM-R-Base-wikipedia-tr-3.5K-s42-LegameX-LoRA-v261010161203'
# LEGAMEX_LANG_CKPT_STEP = 400
# ---
LEGAMEX_TASK_ID = 'alxxtexxr/XLM-R-Base-squad-en-15K-s42-LegameX-LoRA-v261021225519'
LEGAMEX_TASK_CKPT_STEP = 6900
LEGAMEX_LANG_ID = 'alxxtexxr/XLM-R-Base-wikipedia-tr-3.5K-s42-LegameX-LoRA-v261021124754'
LEGAMEX_LANG_CKPT_STEP = 2400
# ---
REF_LORA_ID = 'alxxtexxr/XLM-R-Base-wikipedia-en-2K-s42-LoRA-v260929201303'
REF_LORA_CKPT_STEP = 1000
REF_DELTA_WEIGHT = 1.0  # weight of the reference-LoRA delta (used by 'awr' only)

# Merging configuration.
# The merged weight is W = W_base + SUM_i c_i * d_i, where d_i is each model's RAW
# transfer delta and c_i is a per-model coefficient derived from its gate g_i:
#   'add': c_i = g_i        sum of gated deltas       (SUM g_i*d_i)
#   'avg': c_i = a_i * g_i  scalar-weighted avg       (SUM a_i*g_i*d_i, a=[0.5,0.5])
#   'gna': c_i = g_i / SUM  Gate-Normalized Weighted  (SUM (g_i/SUMg)*d_i, SUM_i c_i=1)
#   'awr': like 'add' PLUS the reference-LoRA delta at REF_DELTA_WEIGHT:
#          W = gate_task*d_task + gate_lang*d_lang + REF_DELTA_WEIGHT * d_ref
ADDITION_TYPE = 'add' # 'add' | 'avg' | 'gna' | 'awr'

assert ADDITION_TYPE in ['add', 'avg', 'gna', 'awr'], \
    f"Invalid addition type: {ADDITION_TYPE}. Must be 'add', 'avg', 'gna', or 'awr'."
if ADDITION_TYPE == 'add':
    addition_weights = [1.0, 1.0]
elif ADDITION_TYPE == 'avg':
    addition_weights = [0.5, 0.5]
elif ADDITION_TYPE == 'awr':
    # The LegameX part is 'add' (weights [1,1] -> c_i = g_i); the reference-LoRA
    # delta is added on top at REF_DELTA_WEIGHT in the merge cell.
    addition_weights = [1.0, 1.0]
else:  # 'gna': coefficients are gate-normalized; scalar weights unused
    addition_weights = None

# Determine the hub merged model ID
assert SRC_LANG in LEGAMEX_TASK_ID and TGT_LANG in LEGAMEX_LANG_ID, \
    "LegameX IDs do not match the specified source language and target language."
model_id_0, model_id_1 = LEGAMEX_TASK_ID.split(SRC_LANG)
model_id_1_0 = model_id_1.split("K-")[-1].split("-v")[0]
hub_merged_model_id = f"{model_id_0}{TGT_LANG}-{model_id_1_0}-{ADDITION_TYPE}-v{datetime.now().strftime("%y%m%d%H%M%S")}"
print(f"Hub merged model ID: {hub_merged_model_id}")


Hub merged model ID: alxxtexxr/XLM-R-Base-squad-tr-s42-LegameX-LoRA-add-v261022143539


# Utilities

In [ ]:
# LoRA utilities
def download_hf_model(
        repo_id: str, 
        ckpt_step: int, 
        max_checkpoints: int = 10_000,
        ckpt_interval: int = 25,
    ) -> tuple[Path, Path]:
    assert len(repo_id.split('/')) == 2, \
        f"Invalid repo_id: {repo_id}. Must be in the format 'username/repo_name'."
    local_dir = Path(repo_id.split('/')[-1])
    ignore_checkpoints = None
    
    if ckpt_step is not None:
        ignore_checkpoints = [
            f'checkpoint-{i}/*' for i in range(0, max_checkpoints, ckpt_interval) 
            if i != ckpt_step
        ]
        ignore_checkpoints += ['runs/*', '**/wandb/*']

    snapshot_download(
        repo_id=repo_id,
        local_dir=local_dir,
        ignore_patterns=ignore_checkpoints,
    )

    ckpt_dir = None
    if ckpt_step is not None:
        ckpt_dir = local_dir / f'checkpoint-{ckpt_step}'
    return local_dir, ckpt_dir

# Gate/LoRA schedule utilities.
# These mirror the schedule math in notebooks/legamex_training_xlmr.ipynb
# (LegamexLora._scheduler_progress / gate_scheduler / gate_offset_scheduler /
# _init_lora_module scaling) so that a merge can reconstruct the EXACT gate
# that was used during training from the checkpoint's legamex_config.json.
def _scheduler_progress(step: int, num_steps: int, scheduler_type: str = 'cosine') -> float:
    """Interpolation factor in [0, 1] following scheduler_type over num_steps.
    Mirrors LegamexLora._scheduler_progress."""
    if num_steps <= 0:
        return 1.0
    
    t = min(max(step / num_steps, 0.0), 1.0)
    
    if scheduler_type == 'linear':
        p = t
    elif scheduler_type == 'cosine':
        p = 0.5 * (1.0 - math.cos(math.pi * t))
    elif scheduler_type == 'quadratic':
        p = t ** 2
    elif scheduler_type == 'cubic':
        p = t ** 3
    elif scheduler_type == 'sqrt':
        p = math.sqrt(t)
    else:
        raise ValueError(
            f"Unknown gate scheduler type: {scheduler_type}. "
            "Must be one of ['linear', 'cosine', 'quadratic', 'cubic', 'sqrt']."
        )
    
    return p

def gate_scheduler_alpha(legamex_config: dict, training_step: int) -> float:
    """Gate warmup factor (0 -> 1 over gate_warmup_steps).
    Mirrors LegamexLora.gate_scheduler()."""
    warmup_steps = legamex_config.get('gate_warmup_steps', 0)
    scheduler_type = legamex_config.get('gate_scheduler_type', 'cosine')
    return _scheduler_progress(training_step, warmup_steps, scheduler_type)

def gate_offset_scheduled(legamex_config: dict, training_step: int) -> float:
    """Scheduled gate-offset term: gate_offset_init annealed to 0.0 over
    gate_offset_steps (constant gate_offset_init when steps <= 0).
    Mirrors LegamexLora.gate_offset_scheduler().
    The LEARNED term (gate.offset in legamex.pt) is separate and added on top,
    i.e. gate_offset_total = gate_offset_scheduled + gate_offset_learned."""
    init = float(legamex_config.get('gate_offset_init', -4.0))
    steps = int(legamex_config.get('gate_offset_steps', 0))
    if steps <= 0:
        return init
    p = _scheduler_progress(training_step, steps, legamex_config.get('gate_scheduler_type', 'cosine'))
    return init + (0.0 - init) * p

def tfr_lora_scaling(legamex_config: dict) -> float:
    """Transfer-LoRA scaling alpha / r (alpha / sqrt(r) when use_rslora).
    Mirrors LegamexLora._init_lora_module's scaling."""
    tfr_lora_config = legamex_config['tfr_lora_config']
    r = tfr_lora_config['r']
    alpha = tfr_lora_config['lora_alpha']
    return alpha / math.sqrt(r) if tfr_lora_config.get('use_rslora', False) else alpha / r


def fix_lora_state_dict(
    lora_state_dict: dict[str, torch.Tensor],
    adapter_config: dict,
) -> OrderedDict[str, torch.Tensor]:
    """Convert a standard PEFT LoRA adapter into raw deltas keyed by base-model
    param names: {<module>.weight: scaling * (B @ A)}.

    The reference LoRA is a plain (ungated) LoRA, so its contribution to a merge
    is simply its delta. Keys are stripped of the 'base_model.model.' prefix so
    they align with AutoModelForQuestionAnswering param names.
    """
    r = adapter_config['r']
    alpha = adapter_config['lora_alpha']
    scaling = alpha / math.sqrt(r) if adapter_config.get('use_rslora', False) else alpha / r

    lora_A: dict[str, torch.Tensor] = {}
    lora_B: dict[str, torch.Tensor] = {}
    for key, value in lora_state_dict.items():
        if key.endswith('.lora_A.weight'):
            lora_A[key[: -len('.lora_A.weight')]] = value
        elif key.endswith('.lora_B.weight'):
            lora_B[key[: -len('.lora_B.weight')]] = value

    assert lora_A and lora_A.keys() == lora_B.keys(), \
        "Reference LoRA must have matching lora_A/lora_B weights."

    delta_state_dict = OrderedDict()
    for module, B in lora_B.items():
        module_name = module.removeprefix('base_model.model.')
        delta_state_dict[f'{module_name}.weight'] = scaling * (B @ lora_A[module])
    return delta_state_dict


# Model

In [6]:
_, legamex_lang_dir = download_hf_model(repo_id=LEGAMEX_LANG_ID, ckpt_step=LEGAMEX_LANG_CKPT_STEP)
legamex_lang_file = legamex_lang_dir / 'legamex.pt'
legamex_lang_state_dict = torch.load(legamex_lang_file, map_location='cpu')
# legamex_config.json holds the merge-time constants (gate_offset_init/steps,
# gate_warmup_steps, tfr lora_alpha/r) that are NOT stored in legamex.pt
with open(legamex_lang_dir / 'legamex_config.json', 'r') as f:
    legamex_lang_config = json.load(f)

_, legamex_task_dir = download_hf_model(repo_id=LEGAMEX_TASK_ID, ckpt_step=LEGAMEX_TASK_CKPT_STEP)
legamex_task_file = legamex_task_dir / 'legamex.pt'
legamex_task_state_dict = torch.load(legamex_task_file, map_location='cpu')
with open(legamex_task_dir / 'legamex_config.json', 'r') as f:
    legamex_task_config = json.load(f)

_, lora_ref_dir = download_hf_model(repo_id=REF_LORA_ID, ckpt_step=REF_LORA_CKPT_STEP)
lora_ref_file = lora_ref_dir / 'adapter_model.safetensors'
lora_ref_state_dict = load_file(lora_ref_file, device='cpu')
with open(lora_ref_dir / 'adapter_config.json', 'r') as f:
    lora_ref_config = json.load(f)

# Raw reference-LoRA deltas keyed by base-model param names (only used by 'awr')
lora_ref_delta_state_dict = fix_lora_state_dict(lora_ref_state_dict, lora_ref_config)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

checkpoint-2400/rng_state.pth:   0%|          | 0.00/14.7k [00:00<?, ?B/s]

checkpoint-2400/legamex.pt:   0%|          | 0.00/11.5M [00:00<?, ?B/s]

checkpoint-2400/scheduler.pt:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

...

trainer_state.json: 0.00B [00:00, ?B/s]

legamex_config.json: 0.00B [00:00, ?B/s]

optimizer_param_names.json: 0.00B [00:00, ?B/s]

trainer_state.json: 0.00B [00:00, ?B/s]

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

In [7]:
def fix_legamex_state_dict(
    state_dict: dict[str, torch.Tensor],
    legamex_config: dict,
    training_step: int,
) -> tuple[OrderedDict[str, torch.Tensor], OrderedDict[str, torch.Tensor]]:
    """Reconstruct each LegameX module's RAW transfer delta and its gate.

    For every wrapped Linear module this recovers, from the checkpoint, the two
    quantities that Gate-Normalized Weighted Averaging combines:

        d_i = tfr_scaling * (B_tfr @ A_tfr)     (raw transfer delta, ungated)
        g_i = gate_scheduler() * sigmoid(B_g @ A_g + offset_sched + offset_learned)

    mirroring LegamexLora's forward / merge_and_unload in
    legamex_training_xlmr.ipynb. The reference (ref) LoRA branch is intentionally
    NOT reconstructed: ref is a shared reference, only the transfer delta
    participates in the merge.

    Returns:
        delta_state_dict: per-module raw delta d_i, plus qa_outputs passthrough
                          (FULL trained head weights, not deltas).
        gate_state_dict:  per-module gate g_i.
    """
    tfr_scaling = tfr_lora_scaling(legamex_config)
    offset_scheduled = gate_offset_scheduled(legamex_config, training_step)
    gate_alpha = gate_scheduler_alpha(legamex_config, training_step)
    print(
        f"tfr_scaling={tfr_scaling}, "
        f"gate_offset_scheduled={offset_scheduled}, "
        f"gate_scheduler_alpha={gate_alpha} "
        f"(training_step={training_step})"
    )
    
    # The scheduled offset shapes the sigmoid, hence the gate RATIOS that GNA
    # depends on, so it must be reconstructed. The warmup alpha is a common
    # factor across all gates and cancels in the normalization w_i = g_i / Σg,
    # but we still warn if the offset has not annealed (it does NOT cancel).
    if int(legamex_config.get('gate_offset_steps', 0)) > 0 and offset_scheduled != 0.0:
        print(
            "Warning: the scheduled gate offset has not annealed to 0 at this "
            "checkpoint; the reconstructed gates (and thus the GNA weights) will "
            "differ from the fully-trained ones."
        )
    
    delta_state_dict = OrderedDict()
    gate_state_dict = OrderedDict()

    lora_tfr_A_weight = None
    lora_tfr_B_weight = None
    gate_A_weight = None
    gate_B_weight = None
    gate_offset = None

    for key, value in state_dict.items():
        key_fixed = key.replace('__DOT__', '.')
        
        if 'lora.tfr.A' in key_fixed:
            lora_tfr_A_weight = value
        elif 'lora.tfr.B' in key_fixed:
            lora_tfr_B_weight = value
        elif 'gate.A' in key_fixed:
            gate_A_weight = value
        elif 'gate.B' in key_fixed:
            gate_B_weight = value
        elif 'gate.offset' in key_fixed:
            gate_offset = value
        elif 'qa_outputs' in key_fixed:
            delta_state_dict[key_fixed] = value
        else:
            continue
            
        if (
            lora_tfr_A_weight is not None and 
            lora_tfr_B_weight is not None and 
            gate_A_weight is not None and 
            gate_B_weight is not None and
            gate_offset is not None
        ):
            # RAW transfer delta d_i WITH the LoRA scaling factor (alpha / r)
            delta_weight = (lora_tfr_B_weight @ lora_tfr_A_weight) * tfr_scaling
            # Gate g_i = sigmoid(raw gate + scheduled offset + learned offset),
            # scaled by the warmup schedule
            gate_weight = torch.sigmoid(
                (gate_B_weight @ gate_A_weight) + offset_scheduled + gate_offset
            ) * gate_alpha
            
            key_base = key_fixed.replace('.gate.', '.lora.').split('.lora.')[0] + '.weight'
            delta_state_dict[key_base] = delta_weight
            gate_state_dict[key_base] = gate_weight
            
            lora_tfr_A_weight = None
            lora_tfr_B_weight = None
            gate_A_weight = None
            gate_B_weight = None
            gate_offset = None
            
    return delta_state_dict, gate_state_dict

legamex_task_delta, legamex_task_gate = fix_legamex_state_dict(
    legamex_task_state_dict, legamex_task_config, LEGAMEX_TASK_CKPT_STEP)
legamex_lang_delta, legamex_lang_gate = fix_legamex_state_dict(
    legamex_lang_state_dict, legamex_lang_config, LEGAMEX_LANG_CKPT_STEP)


tfr_scaling=1.0, gate_offset_scheduled=0.0, gate_scheduler_alpha=1.0 (training_step=6900)
tfr_scaling=1.0, gate_offset_scheduled=0.0, gate_scheduler_alpha=1.0 (training_step=2400)


In [8]:
def normalize_gate_state_dicts(
    gate_state_dicts: list[OrderedDict[str, torch.Tensor]]
) -> list[OrderedDict[str, torch.Tensor]]:
    """'gna': normalize each model's gates so they sum to 1 elementwise.

    w_i = g_i / SUM_j g_j for every weight element, so that SUM_i w_i = 1. This
    is the "gate-normalized" step: the models' gates (which independently lie in
    (0, 1) and may sum to < or > 1) become a convex weighting.
    """
    key_lists = [list(gate_state_dict.keys()) for gate_state_dict in gate_state_dicts]
    assert all(key_list == key_lists[0] for key_list in key_lists), \
        "All gate_state_dicts must have the same keys (and order)."
    
    EPS = 1e-12
    gate_state_dicts_normalized = []
    for key in key_lists[0]:
        values = [gate_state_dict[key] for gate_state_dict in gate_state_dicts]
        sum_ = torch.stack(values).sum(dim=0)
        n_degenerate = int((sum_ <= EPS).sum())
        if n_degenerate:
            print(
                f"Warning: {n_degenerate} weight(s) have ~zero total gate at "
                f"'{key}'; clamping to {EPS} to avoid NaN."
            )
        sum_ = sum_.clamp_min(EPS)
        for i, gate_state_dict in enumerate(gate_state_dicts):
            value_normalized = gate_state_dict[key] / sum_
            if len(gate_state_dicts_normalized) <= i:
                gate_state_dicts_normalized.append(OrderedDict())
            gate_state_dicts_normalized[i][key] = value_normalized
    return gate_state_dicts_normalized


def compute_merge_coefficients(
    addition_type: str,
    gate_state_dicts: list[OrderedDict[str, torch.Tensor]],
    addition_weights: list[float] | None = None,
) -> list[OrderedDict[str, torch.Tensor]]:
    """Per-model merge coefficient c_i, applied to each RAW delta d_i.

    'add' / 'avg' / 'awr': c_i = a_i * g_i  (gated deltas, scalar-weighted; a=[1,1]/[0.5,0.5]/[1,1])
    'gna':         c_i = g_i / SUM  (gate-normalized; SUM_i c_i = 1 elementwise)
    """
    if addition_type in ('add', 'avg', 'awr'):
        assert addition_weights is not None and len(addition_weights) == len(gate_state_dicts), \
            f"'{addition_type}' needs one scalar weight per model."
        return [
            OrderedDict({k: w * gate for k, gate in gate_state_dict.items()})
            for w, gate_state_dict in zip(addition_weights, gate_state_dicts)
        ]
    elif addition_type == 'gna':
        return normalize_gate_state_dicts(gate_state_dicts)
    else:
        raise ValueError(f"Unknown addition type: {addition_type}. Use 'add', 'avg', 'gna', or 'awr'.")


# ORDERING: keep this list in the SAME order as fixed_state_dicts in the merge
# cell below ([task, lang]), so zip() pairs each delta with its own coefficient.
merge_coefficient_state_dicts = compute_merge_coefficients(
    ADDITION_TYPE,
    [legamex_task_gate, legamex_lang_gate],
    addition_weights,
)
merge_coefficient_state_dicts = [
    {k.removeprefix('base_model.'): v for k, v in state_dict.items()}
    for state_dict in merge_coefficient_state_dicts
]


In [9]:
# Load the base model
base_model = AutoModelForQuestionAnswering.from_pretrained(MODEL_ID)

Some weights of XLMRobertaForQuestionAnswering were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# Merge with Weighted Addition / Averaging


In [10]:
# Weighted merge over the models' RAW transfer deltas.
#
#   W_merged = W_base + SUM_i c_i * d_i
#
# where d_i is each model's RAW transfer delta (from fix_legamex_state_dict) and
# c_i is its per-model merge coefficient (from compute_merge_coefficients):
#   'add': c_i = g_i        sum of gated deltas
#   'avg': c_i = a_i * g_i  scalar-weighted average of gated deltas
#   'gna': c_i = g_i / SUMg Gate-Normalized Weighted Averaging (convex combination)
#
# This mirrors the sequential PEFT merge in notebooks/lora_addition_xlmr.ipynb
# (W <- W + w * (B @ (w-scaled) A)), but with the deltas precomputed and the
# per-model coefficient derived from the learned gate.
#
# Ordering: fixed_state_dicts and merge_coefficient_state_dicts are both
# [task, lang], so zip() pairs each delta with its own coefficient.
fixed_state_dicts = [
    legamex_task_delta,
    legamex_lang_delta,
]

# Defensive normalization: strip a 'base_model.' prefix if present
# (current checkpoints do not use one; older versions did).
fixed_state_dicts = [
    {k.removeprefix('base_model.'): v for k, v in state_dict.items()}
    for state_dict in fixed_state_dicts
]

current_model = base_model
for delta_state_dict, coeff_state_dict in zip(fixed_state_dicts, merge_coefficient_state_dicts):
    n_deltas, n_heads = 0, 0
    with torch.no_grad():
        for name, param in current_model.named_parameters():
            if name not in delta_state_dict:
                continue
            value = delta_state_dict[name].to(device=param.device, dtype=param.dtype)
            if 'qa_outputs' in name:
                # qa_outputs entries are FULL trained weights, not deltas:
                # the base XLM-R QA head is randomly initialized, so we copy
                # the trained head instead of adding it on top of noise.
                # (Only the squad/task LegameX provides a QA head; the
                #  wikipedia lang LegameX is FEATURE_EXTRACTION and has none.)
                param.data.copy_(value)
                n_heads += 1
            else:
                # W += c_i * d_i   (c_i is the per-model merge coefficient)
                coeff = coeff_state_dict[name].to(device=param.device, dtype=param.dtype)
                param.data += coeff * value
                n_deltas += 1
    print(f"[{ADDITION_TYPE}] model: added {n_deltas} deltas (coefficient c_i), copied {n_heads} qa_outputs tensors")

# 'awr': add the reference-LoRA delta on top of the gated LegameX deltas.
# The reference LoRA is a plain (ungated) LoRA, so its contribution is just its
# delta scaled by REF_DELTA_WEIGHT. It targets only attention/FFN modules (NOT
# qa_outputs), so the task model's trained QA head is preserved.
if ADDITION_TYPE == 'awr':
    n_ref = 0
    with torch.no_grad():
        for name, param in current_model.named_parameters():
            if name not in lora_ref_delta_state_dict:
                continue
            value = lora_ref_delta_state_dict[name].to(device=param.device, dtype=param.dtype)
            param.data += REF_DELTA_WEIGHT * value
            n_ref += 1
    print(f"[awr] added {n_ref} reference-LoRA deltas at REF_DELTA_WEIGHT={REF_DELTA_WEIGHT}")

merged_model = current_model


[add] model: added 72 deltas (coefficient c_i), copied 2 qa_outputs tensors
[add] model: added 72 deltas (coefficient c_i), copied 0 qa_outputs tensors


In [11]:
# Sanity check
for i in range(11):
    print(f"Merged model layer-{i} attention value weight norm:", merged_model.roberta.encoder.layer[i].attention.self.value.weight.norm().item())
print()
print("Merged model qa_outputs weight norm:", merged_model.qa_outputs.weight.norm().item())
print("Merged model qa_outputs bias norm:", merged_model.qa_outputs.bias.norm().item())

# 'gna' invariant: the merge coefficients must sum to 1 at every weight element,
# i.e. the merged delta is a true convex combination of the two models' deltas.
# ('add'/'avg' coefficients sum to SUM g_i / 0.5*SUM g_i, which is not 1 in general.)
if ADDITION_TYPE == 'gna':
    first_key = next(iter(merge_coefficient_state_dicts[0]))
    coeff_sum = (merge_coefficient_state_dicts[0][first_key]
                 + merge_coefficient_state_dicts[1][first_key])
    print()
    print(f"GNA coefficient sum (task+lang) at '{first_key}': "
          f"min={coeff_sum.min().item():.6f}, max={coeff_sum.max().item():.6f}  (expect 1.0)")

# 'awr': report the reference-LoRA delta coverage (should exclude qa_outputs)
if ADDITION_TYPE == 'awr':
    ref_keys = list(lora_ref_delta_state_dict.keys())
    print()
    print(f"'awr' reference-LoRA deltas: {len(ref_keys)} tensors, "
          f"qa_outputs present: {any('qa_outputs' in k for k in ref_keys)}")
    print(f"e.g. {ref_keys[0]}")


Merged model layer-0 attention value weight norm: 28.369274139404297
Merged model layer-1 attention value weight norm: 28.85371971130371
Merged model layer-2 attention value weight norm: 30.166757583618164
Merged model layer-3 attention value weight norm: 36.58069610595703
Merged model layer-4 attention value weight norm: 38.33047103881836
Merged model layer-5 attention value weight norm: 41.677734375
Merged model layer-6 attention value weight norm: 38.23060989379883
Merged model layer-7 attention value weight norm: 39.02166748046875
Merged model layer-8 attention value weight norm: 39.22397232055664
Merged model layer-9 attention value weight norm: 34.93556594848633
Merged model layer-10 attention value weight norm: 31.137540817260742

Merged model qa_outputs weight norm: 1.1597857475280762
Merged model qa_outputs bias norm: 0.005278421100229025


In [12]:
# Upload the merged model to Hugging Face
merged_model.push_to_hub(hub_merged_model_id)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.push_to_hub(hub_merged_model_id)

print(f"Merged model uploaded to: https://huggingface.co/{hub_merged_model_id}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...h7z3jrf/model.safetensors:   2%|2         | 23.1MB / 1.11GB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...d/sentencepiece.bpe.model: 100%|##########| 5.07MB / 5.07MB            

  ...mphmxhio3d/tokenizer.json: 100%|##########| 17.1MB / 17.1MB            

Merged model uploaded to: https://huggingface.co/alxxtexxr/XLM-R-Base-squad-tr-s42-LegameX-LoRA-add-v261022143539
